In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm#tqdm库显示进度条
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# 定义实用函数
def R2_cal(y, y_pred):
    """ 计算R2得分 """
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum((y - np.mean(y))**2)
    return r2

def relu(x):
    """ 定义ReLU激活函数 """
    return np.maximum(0, x)

def relu_grad(x):
    """ 计算ReLU函数的梯度 """
    grad = np.zeros_like(x)
    grad[x > 0] = 1
    return grad

def mean_squared_error(y, t):
    """ 计算均方误差 """
    return 0.5 * np.sum((y - t) ** 2)

# 定义四层隐藏层神经网络类
class FourLayerNet:

    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, hidden4_size, output_size, weight_init_std=0.01):
        """ 初始化网络参数 """
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden1_size)
        self.params['b1'] = np.zeros(hidden1_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden1_size, hidden2_size)
        self.params['b2'] = np.zeros(hidden2_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden2_size, hidden3_size)
        self.params['b3'] = np.zeros(hidden3_size)
        self.params['W4'] = weight_init_std * np.random.randn(hidden3_size, hidden4_size)
        self.params['b4'] = np.zeros(hidden4_size)
        self.params['W5'] = weight_init_std * np.random.randn(hidden4_size, output_size)
        self.params['b5'] = np.zeros(output_size)

    def predict(self, x):
        """ 前向传播预测 """
        W1, W2, W3, W4, W5 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4'], self.params['W5']
        b1, b2, b3, b4, b5 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4'], self.params['b5']
    
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        z2 = relu(a2)
        a3 = np.dot(z2, W3) + b3
        z3 = relu(a3)
        a4 = np.dot(z3, W4) + b4
        z4 = relu(a4)
        a5 = np.dot(z4, W5) + b5
        y = a5  # 输出层没有激活函数，用于回归
        
        return y
        
    def loss(self, x, t):
        """ 计算损失 """
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        """ 计算均方误差（用于衡量模型的性能） """
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  # 使用均方误差(MSE)来衡量回归模型的性能
        return error
        
    def gradient(self, x, t):
        """ 计算梯度（反向传播） """
        W1, W2, W3, W4, W5 = self.params['W1'], self.params['W2'], self.params['W3'], self.params['W4'], self.params['W5']
        b1, b2, b3, b4, b5 = self.params['b1'], self.params['b2'], self.params['b3'], self.params['b4'], self.params['b5']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        z1 = relu(a1)
        a2 = np.dot(z1, W2) + b2
        z2 = relu(a2)
        a3 = np.dot(z2, W3) + b3
        z3 = relu(a3)
        a4 = np.dot(z3, W4) + b4
        z4 = relu(a4)
        a5 = np.dot(z4, W5) + b5
        y = a5
        
        # backward
        dy = (y - t) / batch_num
        grads['W5'] = np.dot(z4.T, dy)
        grads['b5'] = np.sum(dy, axis=0)
        
        dz4 = np.dot(dy, W5.T) * relu_grad(a4)
        grads['W4'] = np.dot(z3.T, dz4)
        grads['b4'] = np.sum(dz4, axis=0)
        
        dz3 = np.dot(dz4, W4.T) * relu_grad(a3)
        grads['W3'] = np.dot(z2.T, dz3)
        grads['b3'] = np.sum(dz3, axis=0)
        
        dz2 = np.dot(dz3, W3.T) * relu_grad(a2)
        grads['W2'] = np.dot(z1.T, dz2)
        grads['b2'] = np.sum(dz2, axis=0)
        
        dz1 = np.dot(dz2, W2.T) * relu_grad(a1)
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)

        return grads

In [ ]:
# 假设数据已经加载到一个DataFrame `data`中
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

# 初始化参数
start_year_train = 1957  
start_year_val = 1975  
start_year_test = 1987  
end_year_test = 2016

pred_y_list = []
true_y_list = []

# 迭代处理每个测试年份
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # 定义训练、验证和测试的日期范围
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    # 打印日期范围以便调试
    print(f'训练开始日期: {start_date_train.strftime("%Y-%m")}')
    print(f'训练结束日期: {end_date_train.strftime("%Y-%m")}')
    print(f'验证开始日期: {start_date_val.strftime("%Y-%m")}')
    print(f'验证结束日期: {end_date_val.strftime("%Y-%m")}')
    print(f'测试开始日期: {start_date_test.strftime("%Y-%m")}')
    print(f'测试结束日期: {end_date_test.strftime("%Y-%m")}')

    # 根据日期选择相应的训练、验证和测试数据
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # 提取特征和目标变量
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_train = train_data['RET'].values.reshape(-1, 1)
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values.reshape(-1, 1)
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values.reshape(-1, 1)
    
    # 初始化神经网络
    net_work = FourLayerNet(input_size=X_train.shape[1], hidden1_size=64, hidden2_size=32, hidden3_size=16, hidden4_size=8, output_size=1)

    # 训练参数
    n_epochs = 10
    
    train_size = X_train.shape[0]
    batch_size = 100
    learning_rate = 0.02
    patience = 20
    lr_decay = 0.9

    best_model_state = None
    best_loss = float('inf')
    counter = 0

    # 训练循环
    for epoch in range(n_epochs):
        for i in range(0, train_size, batch_size):
            xi = X_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]

            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4', 'W5', 'b5'):
                net_work.params[key] -= learning_rate * grad[key]
        
        val_loss = net_work.accuracy(X_val, y_val)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                learning_rate *= lr_decay
                counter = 0

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(X_train, y_train)
            test_loss = net_work.accuracy(X_val, y_val)
            print(f"Epoch {epoch}: 训练误差: {train_loss}, 验证误差: {val_loss}")

    # 加载最佳模型
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4', 'W5', 'b5'):
        net_work.params[key] = best_model_state[key]

    # 预测并评估
    y_pred = net_work.predict(X_test)
    r2_score = R2_cal(y_test, y_pred)
    print(f"Year {year}: R2 得分 = {r2_score}")

    pred_y_list.append(y_pred)
    true_y_list.append(y_test)

# 转换预测和真实值的列表为数组
pred_y_list = np.vstack(pred_y_list)
true_y_list = np.vstack(true_y_list)

# 评估整体性能
overall_r2_score = R2_cal(true_y_list, pred_y_list)
print(f"整体 R2 得分 = {overall_r2_score}")

构建了一个包含四层隐藏层的神经网络，每层分别包含64、32、16和8个神经元。我们使用ReLU作为激活函数，并通过随机梯度下降(SGD)进行训练，同时引入学习率缩减（Learning Rate Decay）进行正则化。当验证误差未能改善时，逐渐减小学习率，以提高模型的稳定性和性能。
